In [0]:
import sys

repo_path = "/Workspace/databricks-genai-rag-agent-portfolio"

if repo_path not in sys.path:

    sys.path.insert(0, repo_path)

from src.trend_agent import get_most_degraded_table

In [0]:
from pyspark.sql.functions import col

trend_df = spark.table("workspace.default.gold_quality_trends")

display(trend_df)

In [0]:
def get_root_cause(table_name):

    rule_df = spark.table("workspace.default.gold_quality_by_rule")

    failed_rules = (
        rule_df
        .filter(col("table_name") == table_name)
        .orderBy(col("total_failure_percentage").desc())
        .collect()
    )

    root_causes = []

    for rule in failed_rules:

        rule_name = rule["rule"]

        if rule_name == "not_null":
            root_causes.append(
                "Missing mandatory fields increased significantly."
            )

        elif rule_name == "unique":
            root_causes.append(
                "Duplicate records detected in source data."
            )

        elif rule_name == "regex":
            root_causes.append(
                "Format validation failures increased."
            )

        elif rule_name == "accepted_values":
            root_causes.append(
                "Unexpected domain values appeared."
            )

        else:
            root_causes.append(
                f"Failure spike detected in rule {rule_name}."
            )

    return root_causes

In [0]:
def build_root_cause_response(question):

    degraded_table = get_most_degraded_table(spark)

    table_name = degraded_table["table_name"]

    causes = get_root_cause(table_name)

    response = f"""
Question:
{question}

Table Under Investigation:
{table_name}

Quality Change:
{degraded_table["quality_score_change"]}

Likely Root Causes:
"""

    for cause in causes:
        response += f"\n- {cause}"

    response += """

Recommended Actions:

- Review latest ingestion runs.
- Compare source files with previous loads.
- Check schema changes.
- Review duplicate records.
- Review null value increases.
- Monitor quality trend continuously.

Investigation completed by Root Cause Agent.
"""

    return response

In [0]:
print(
    build_root_cause_response(
        "Why did quality drop?"
    )
)

In [0]:
from src.question_router import route_question

print(route_question(spark, "Which table has the worst data quality?"))

print(route_question(spark, "Which table quality degraded the most?"))